# N14 — Agent2Agent (A2A): Interoperabilidade entre Agentes

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>

## 0.1 Pré-requisitos

Para acompanhar este notebook, é esperado que você já tenha:

- construído agentes com `create_agent` e ferramentas (Notebook 4/7);
- visto o **Model Context Protocol** no Notebook 5 — vamos contrastar A2A com MCP o tempo todo;
- visto os quatro padrões de comunicação do Notebook 13 (`memory`, `relay`, `report`, `debate`) — A2A resolve um problema diferente do deles, e a comparação é o fio condutor da aula;
- noção básica de cliente/servidor HTTP e JSON (não precisa saber JSON-RPC de antemão).

## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Explicar que problema o A2A resolve, e por que ele **não** é redundante com o MCP nem com os padrões do Notebook 13.
2. Descrever as peças centrais do protocolo: **AgentCard**, **Message**/**Part**, **Task** e o ciclo de vida de uma task (`TaskState`).
3. Implementar um **AgentExecutor** que expõe um agente LangChain do curso como um serviço A2A real, com o `a2a-sdk`.
4. Descobrir um agente remoto pelo seu AgentCard e conversar com ele por um cliente A2A, em modo síncrono e em streaming.
5. Cancelar uma task em andamento e entender por que isso importa em um protocolo pensado para agentes de longa duração.
6. Compor um pequeno sistema multiagente onde cada agente é um **processo independente**, replicando o padrão *report* do Notebook 13 — agora entre processos, não dentro de um único grafo.

## 0.3 Mapa do notebook

1. Por que A2A — o problema que MCP e os padrões do Notebook 13 não resolvem.
2. Anatomia mínima do protocolo: AgentCard, Message, Task (sem rede, só para fixar o formato).
3. Um servidor A2A de verdade: `AgentExecutor`, `TaskUpdater`, `AgentCard`, subindo com `uvicorn`.
4. Um cliente A2A de verdade: descoberta do AgentCard e `send_message`.
5. Streaming: acompanhando o ciclo de vida de uma task em tempo real.
6. Cancelando uma task em andamento.
7. Compondo agentes via A2A — o padrão *report* do Notebook 13, agora entre processos.
8. A2A vs. MCP vs. os padrões intra-processo do Notebook 13.
9. Exercícios, resumo e referências.

## 0.4 Contexto usado nos exemplos

Continuamos com o mesmo elenco de agentes dos Notebooks 7 e 13 — `agente_tutor`, `agente_secretaria`, `agente_pesquisa` — mas agora cada agente que expusermos via A2A vai rodar como um **serviço HTTP independente**, na própria máquina (`127.0.0.1`), em uma porta própria. Nada impediria que esse serviço rodasse em outra máquina, em outra linguagem, ou dentro de outra empresa: é exatamente esse desacoplamento que o A2A padroniza.

**Sobre a Eagle:** o framework usado nos Notebooks 12 e 13 não tem suporte a A2A hoje — conferimos isso procurando por `"a2a"` no código-fonte do pacote `dev-tools-eagle` instalado (`2.1.6.10`) e não há nenhuma ocorrência. Isso não é uma limitação do LangGraph nem do A2A; é só um framework de orquestração intra-processo que ainda não integrou um protocolo de interoperabilidade entre processos. Por isso este notebook usa o SDK oficial (`a2a-sdk`) diretamente, sem seção Eagle.

---

## 0.5 Preparação do ambiente

O `a2a-sdk` está em desenvolvimento ativo. Entre quando este material foi escrito e agora, a versão majoritária no PyPI mudou de `0.3.x` para `1.x` — e a `1.x` **removeu** `A2AStarletteApplication`, a forma mais simples (e mais documentada nos tutoriais oficiais) de montar um servidor A2A. Por isso fixamos a versão abaixo. Se você quiser investigar a API nova, isso vira o Exercício sobre "A2A na versão mais nova" ao final do notebook.

In [1]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
%pip install -U langgraph langchain langchain-ollama pydantic httpx uvicorn "a2a-sdk==0.3.26"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Imports usados ao longo da aula.

In [2]:
import asyncio
import json
import threading
import time
import uuid
import warnings
from typing import Literal

# A2AClient está marcado como deprecated nesta versão do SDK (seção 4 explica por quê).
# Silenciamos só esse aviso específico para não poluir toda saída do notebook com ele repetido.
warnings.filterwarnings("ignore", message="A2AClient is deprecated.*", category=DeprecationWarning)

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

import httpx
import uvicorn

from a2a.client import A2ACardResolver, A2AClient, create_text_message_object
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    CancelTaskRequest,
    GetTaskRequest,
    MessageSendParams,
    Role,
    SendMessageRequest,
    SendStreamingMessageRequest,
    Task,
    TaskIdParams,
    TaskQueryParams,
    TaskState,
)
from a2a.utils import new_agent_text_message

Funções auxiliares de exibição. `print_json` já apareceu no Notebook 5; `subir_servidor_a2a` é nova — sobe um app A2A numa thread separada, para o servidor conviver com o resto do notebook no mesmo processo Python. Reexecutar a célula que a chama derruba a instância antiga antes de subir a nova, então religar um servidor na mesma porta não quebra por "endereço já em uso".

In [3]:
def print_json(value) -> None:
    """Imprime objetos Python (ou modelos Pydantic) como JSON formatado."""
    if hasattr(value, "model_dump"):
        value = value.model_dump(exclude_none=True, by_alias=True)
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


_servidores_a2a: dict[int, uvicorn.Server] = {}

# Todos os servidores A2A deste notebook rodam numa única thread/event loop
# compartilhada -- não uma por servidor. O motivo não é só simplicidade: o
# cliente Ollama assíncrono dentro de cada agente (ver "ainvoke" no
# AgentExecutor) prende recursos internos (locks asyncio) ao event loop que o
# usou primeiro. Threads diferentes, cada uma com seu próprio
# `asyncio.run(...)`, dão event loops diferentes -- e o segundo agente a usar
# o mesmo `llm` compartilhado (seção 0.4) quebraria com "bound to a different
# event loop". Um loop só, para todos os servidores, evita isso.
_loop_a2a = asyncio.new_event_loop()
threading.Thread(target=_loop_a2a.run_forever, daemon=True).start()


def subir_servidor_a2a(app, porta: int) -> uvicorn.Server:
    """Sobe um app A2A (Starlette/ASGI) no event loop compartilhado, na porta dada."""
    antigo = _servidores_a2a.get(porta)
    if antigo is not None:
        antigo.should_exit = True
        time.sleep(0.3)

    config = uvicorn.Config(app, host="127.0.0.1", port=porta, log_level="warning")
    server = uvicorn.Server(config)
    asyncio.run_coroutine_threadsafe(server.serve(), _loop_a2a)

    # Espera ativamente o socket responder, em vez de um sleep fixo -- mais robusto
    # sob máquina compartilhada/carregada do que torcer para 1s ter sido suficiente.
    url_card = f"http://127.0.0.1:{porta}/.well-known/agent-card.json"
    with httpx.Client(timeout=1) as probe:
        for _ in range(50):
            try:
                if probe.get(url_card).status_code == 200:
                    break
            except httpx.HTTPError:
                pass
            time.sleep(0.2)
        else:
            raise RuntimeError(f"Servidor não respondeu em http://127.0.0.1:{porta} a tempo.")

    _servidores_a2a[porta] = server
    return server


def texto_da_resposta(task: Task) -> str:
    """Extrai o texto da mensagem final de uma task A2A completada."""
    if task.status.message is None:
        return "(a task não trouxe mensagem final)"
    return "".join(part.root.text for part in task.status.message.parts if part.root.kind == "text")

Inicializamos o LLM (mesma configuração dos notebooks anteriores) e reconstruímos os três agentes especialistas do curso — mesmo cenário do Notebook 13.

In [4]:
llm = init_chat_model(model="ollama:qwen3:14b", base_url="http://localhost:11500")

In [5]:
MODULOS_CURSO = {
    "A4": {"tema": "Ferramentas e create_agent", "duracao_horas": 6},
    "A5": {"tema": "Memória em LangGraph", "duracao_horas": 5},
    "A6": {"tema": "Memória Reflexiva", "duracao_horas": 4},
    "A7": {"tema": "Arquiteturas Multiagente", "duracao_horas": 6},
    "A8": {"tema": "Padrões de Comunicação entre Agentes", "duracao_horas": 6},
}

PRAZOS_CURSO = {
    "matricula": "até 20/08/2026",
    "trancamento": "até 30/09/2026",
    "certificado": "emitido em até 10 dias úteis após a conclusão do último módulo",
}

BASE_CONHECIMENTO = {
    "agentcard": "Um AgentCard é o documento público que descreve um agente A2A: nome, descrição, skills e onde falar com ele.",
    "taskstate": "TaskState é o estado de uma task A2A: submitted, working, input-required, completed, failed, canceled, entre outros.",
    "jsonrpc": "JSON-RPC 2.0 é o formato de mensagem usado pelo transporte padrão do A2A: campos jsonrpc, id, method, params/result.",
}


def consultar_modulo(codigo_modulo: str) -> dict:
    """Retorna o tema e a carga horária de um módulo do curso, dado seu código (ex.: A6)."""
    return MODULOS_CURSO.get(codigo_modulo, {"erro": "módulo não encontrado"})


def consultar_prazo(tipo: str) -> str:
    """Consulta prazos administrativos do curso. tipo pode ser: matricula, trancamento ou certificado."""
    return PRAZOS_CURSO.get(tipo, "Prazo não encontrado. Fale diretamente com a secretaria.")


def emitir_declaracao(aluno: str, modulo_atual: str) -> str:
    """Emite uma declaração simples de matrícula ativa para o aluno."""
    return f"Declaração emitida: {aluno} está matriculado(a) e cursando o módulo {modulo_atual}."


def buscar_conceito(termo: str) -> str:
    """Busca a definição de um conceito técnico (LangGraph, A2A, MCP...) na base de conhecimento do curso."""
    termo_normalizado = termo.lower().strip()
    for chave, definicao in BASE_CONHECIMENTO.items():
        if chave in termo_normalizado or termo_normalizado in chave:
            return definicao
    return "Conceito não encontrado na base de conhecimento do curso."


agente_tutor = create_agent(
    model=llm,
    tools=[consultar_modulo],
    system_prompt=(
        "Você é o tutor técnico do curso de LangGraph. "
        "Responda apenas dúvidas sobre o conteúdo e a carga horária dos módulos."
    ),
)

agente_secretaria = create_agent(
    model=llm,
    tools=[consultar_prazo, emitir_declaracao],
    system_prompt=(
        "Você é a secretaria do curso de LangGraph. "
        "Responda apenas dúvidas administrativas: prazos, matrícula, certificado."
    ),
)

agente_pesquisa = create_agent(
    model=llm,
    tools=[buscar_conceito],
    system_prompt=(
        "Você é o agente de pesquisa conceitual do curso de LangGraph. "
        "Responda definições de conceitos técnicos gerais, sem falar de módulos específicos."
    ),
)

---

# 1. Por que A2A — o problema que faltava resolver

O Notebook 5 (MCP) resolveu um problema: como um agente fala com **ferramentas e dados** de forma padronizada, em vez de uma integração ad hoc para cada API. O Notebook 13 resolveu outro: como **vários agentes dentro do mesmo grafo Python** se coordenam — via memória compartilhada, repasse em cadeia, um supervisor central, ou debate.

Nenhum dos dois resolve um terceiro problema, cada vez mais comum: dois agentes que **não compartilham processo, nem código, nem necessariamente o mesmo framework ou a mesma empresa** — mas que ainda assim precisam conversar. Exemplos:

- o agente de reservas de voo da sua empresa precisa pedir a um agente de outra empresa que verifique a disponibilidade de um hotel;
- um agente construído em LangGraph precisa delegar uma tarefa a um agente construído em outra stack qualquer, hospedado por outro time;
- um orquestrador quer descobrir, em tempo de execução, **quais agentes existem** e **o que cada um sabe fazer**, sem ter sido programado de antemão com esse conhecimento.

```text
MCP     agente ⇄ [servidor MCP]           (agente fala com uma FERRAMENTA/dado)

N13     agente_a ⇄ agente_b ⇄ agente_c    (agentes no MESMO grafo, MESMO processo Python)
        (memory / relay / report / debate)

A2A     Agente A  <--- HTTP + JSON-RPC --->  Agente B
        processo 1, framework/linguagem 1     processo 2, framework/linguagem 2
        (cada lado é uma CAIXA-PRETA para o outro: só o AgentCard é público)
```

**Agent2Agent (A2A)** é um protocolo aberto (lançado pelo Google em 2025, hoje sob a Linux Foundation) para esse terceiro caso: comunicação **entre** sistemas de agentes, através de um contrato de rede padronizado, independente de framework. Um agente A2A expõe um **AgentCard** (o que ele é, e onde falar com ele) e conversa por **tasks** e **messages**, sobre HTTP.

A pergunta que organiza este notebook, no mesmo espírito da tabela do Notebook 13: qual problema cada camada resolve, e onde ela para.

| | MCP (N5) | Padrões do N13 | A2A (este notebook) |
|---|---|---|---|
| Quem fala com quem | agente ⇄ ferramenta/dado | agente ⇄ agente, mesmo processo | agente ⇄ agente, processos/organizações diferentes |
| Compartilham código? | não precisa | sim — mesmo grafo, mesmo `state` | não — só o contrato do protocolo |
| Unidade de trabalho | chamada de tool | nó de um `StateGraph` | **task**, com ciclo de vida próprio |
| Descoberta | `tools/list` no servidor MCP | nenhuma — o grafo já sabe quem existe | **AgentCard**, publicado num endpoint bem conhecido |

---

# 2. Anatomia mínima do protocolo

Antes de subir um servidor de verdade, vale fixar o formato das três peças centrais — sem rede nenhuma, só olhando os dicionários que trafegam. É o mesmo espírito da simulação didática do Notebook 5 para o MCP.

## 2.1 AgentCard

O **AgentCard** é o "cartão de visitas" público de um agente: o que ele é, o que sabe fazer (`skills`), e onde falar com ele (`url`). Um cliente A2A busca esse documento **antes** de trocar qualquer mensagem — é assim que a descoberta acontece, sem precisar programar de antemão quem existe.

In [6]:
exemplo_agent_card = {
    "name": "Secretaria",
    "description": "Agente administrativo do curso: prazos, matrícula, certificado.",
    "url": "http://127.0.0.1:8901/",
    "version": "0.1.0",
    "protocolVersion": "0.3.0",
    "defaultInputModes": ["text"],
    "defaultOutputModes": ["text"],
    "capabilities": {"streaming": True},
    "skills": [
        {
            "id": "secretaria",
            "name": "Secretaria",
            "description": "Responde dúvidas administrativas do curso.",
            "tags": ["curso", "administrativo"],
        }
    ],
}

print_json(exemplo_agent_card)

{
  "name": "Secretaria",
  "description": "Agente administrativo do curso: prazos, matrícula, certificado.",
  "url": "http://127.0.0.1:8901/",
  "version": "0.1.0",
  "protocolVersion": "0.3.0",
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "capabilities": {
    "streaming": true
  },
  "skills": [
    {
      "id": "secretaria",
      "name": "Secretaria",
      "description": "Responde dúvidas administrativas do curso.",
      "tags": [
        "curso",
        "administrativo"
      ]
    }
  ]
}


## 2.2 Message e Task

Uma **Message** carrega conteúdo entre as partes (texto, mas também dados estruturados ou arquivos, via `parts`). Uma **Task** é o que se cria quando você pede a um agente A2A para fazer algo — ela tem um `id`, um `status` (com o `TaskState` atual) e, ao final, a mensagem de resposta.

O transporte padrão é **JSON-RPC 2.0** sobre HTTP: o cliente chama o método `message/send` com uma `Message`, e o servidor devolve uma `Task`.

In [7]:
exemplo_message_send_request = {
    "jsonrpc": "2.0",
    "id": "req-1",
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "Qual o prazo de matrícula?"}],
            "messageId": "msg-1",
        }
    },
}

exemplo_task_response = {
    "jsonrpc": "2.0",
    "id": "req-1",
    "result": {
        "id": "task-1",
        "contextId": "ctx-1",
        "kind": "task",
        "status": {
            "state": "completed",
            "message": {
                "role": "agent",
                "parts": [{"kind": "text", "text": "O prazo de matrícula é até 20/08/2026."}],
                "messageId": "msg-2",
            },
        },
    },
}

print_json(exemplo_message_send_request)
print()
print_json(exemplo_task_response)

{
  "jsonrpc": "2.0",
  "id": "req-1",
  "method": "message/send",
  "params": {
    "message": {
      "role": "user",
      "parts": [
        {
          "kind": "text",
          "text": "Qual o prazo de matrícula?"
        }
      ],
      "messageId": "msg-1"
    }
  }
}

{
  "jsonrpc": "2.0",
  "id": "req-1",
  "result": {
    "id": "task-1",
    "contextId": "ctx-1",
    "kind": "task",
    "status": {
      "state": "completed",
      "message": {
        "role": "agent",
        "parts": [
          {
            "kind": "text",
            "text": "O prazo de matrícula é até 20/08/2026."
          }
        ],
        "messageId": "msg-2"
      }
    }
  }
}


## 2.3 O ciclo de vida de uma task (`TaskState`)

Diferente de uma chamada de tool no MCP (que termina na hora), uma task A2A é pensada para trabalho que pode **demorar**, precisar de **mais input do usuário**, ou ser **cancelada** no meio do caminho. `TaskState` é a máquina de estados que formaliza isso:

```text
submitted → working → completed
               │  └──────────────→ failed
               │  └──────────────→ canceled
               └────────────────→ input-required → working → ...
```

| Estado | Significado |
|---|---|
| `submitted` | a task foi recebida, ainda não começou a rodar |
| `working` | o agente está processando |
| `input-required` | o agente parou e precisa de mais informação do solicitante |
| `completed` | terminou com sucesso; a resposta está em `status.message` |
| `failed` | terminou com erro |
| `canceled` | foi cancelada (pelo cliente ou pelo servidor) antes de terminar |

As próximas seções sobem um servidor A2A de verdade e observam esses estados acontecendo.

---

# 3. Um servidor A2A de verdade

Para expor um agente via A2A, o `a2a-sdk` pede três peças:

1. um **`AgentExecutor`** — a classe que você implementa, com a lógica de verdade do agente;
2. um **`AgentCard`** — o documento de descoberta (seção 2.1, agora real);
3. uma aplicação ASGI que junta as duas (`A2AStarletteApplication`), servida por um servidor HTTP (`uvicorn`).

## 3.1 `AgentExecutor`: onde a lógica do agente mora

`AgentExecutor` é uma interface com dois métodos: `execute` (rodar a task) e `cancel` (cancelar uma task em andamento — seção 6). Dentro de `execute`, você lê o pedido do solicitante em `context.get_user_input()` e publica atualizações de status através de um `TaskUpdater`, que empacota os eventos certos (`submitted`, `working`, `completed`, ...) para a fila de eventos do servidor.

Aqui empacotamos o `agente_secretaria` — o mesmo `create_agent` da seção 0.4, sem nenhuma alteração nele.

In [8]:
class AgenteLangChainExecutor(AgentExecutor):
    """Empacota um agente LangChain (create_agent) como um AgentExecutor A2A."""

    def __init__(self, agente, rotulo: str):
        self._agente = agente
        self._rotulo = rotulo

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        pedido = context.get_user_input()
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)

        if context.current_task is None:
            await updater.submit()
        await updater.start_work()

        resultado = await self._agente.ainvoke({"messages": [{"role": "user", "content": pedido}]})
        resposta = resultado["messages"][-1].content

        await updater.complete(message=new_agent_text_message(resposta))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.update_status(TaskState.canceled, final=True)

## 3.2 `AgentCard`: a versão real da seção 2.1

Mesmos campos do dicionário de exemplo, agora como o modelo Pydantic que o SDK espera. `url` é onde o servidor vai escutar; `skills` é a lista do que esse agente sabe fazer (um agente pode ter mais de uma skill).

In [9]:
PORTA_SECRETARIA = 8901

secretaria_card = AgentCard(
    name="Secretaria",
    description="Agente administrativo do curso: prazos, matrícula, certificado.",
    url=f"http://127.0.0.1:{PORTA_SECRETARIA}/",
    version="0.1.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[
        AgentSkill(
            id="secretaria",
            name="Secretaria",
            description="Responde dúvidas administrativas do curso.",
            tags=["curso", "administrativo"],
        )
    ],
)

## 3.3 Montando e subindo o servidor

`DefaultRequestHandler` é a peça do SDK que sabe falar JSON-RPC e traduzir isso em chamadas ao seu `AgentExecutor` — incluindo gerenciar o ciclo de vida da task via um `TaskStore` (aqui, em memória; um `DatabaseTaskStore` existe para persistir tasks entre reinícios). `A2AStarletteApplication` junta handler e AgentCard numa aplicação ASGI, que sobe com `uvicorn` na thread auxiliar definida na seção 0.5.

In [10]:
handler_secretaria = DefaultRequestHandler(
    agent_executor=AgenteLangChainExecutor(agente_secretaria, "Secretaria"),
    task_store=InMemoryTaskStore(),
)
app_secretaria = A2AStarletteApplication(agent_card=secretaria_card, http_handler=handler_secretaria).build()

subir_servidor_a2a(app_secretaria, PORTA_SECRETARIA)
print(f"Servidor da Secretaria no ar em http://127.0.0.1:{PORTA_SECRETARIA}")

Servidor da Secretaria no ar em http://127.0.0.1:8901


---

# 4. Um cliente A2A de verdade

Do lado de quem consome, o fluxo é: (1) descobrir o AgentCard, (2) montar um cliente a partir dele, (3) mandar uma mensagem. O `A2ACardResolver` busca o AgentCard no endpoint bem conhecido `/.well-known/agent-card.json` — é assim que um cliente descobre um agente que nunca viu antes, sabendo só a URL base.

In [11]:
async with httpx.AsyncClient(timeout=30) as http_client:
    resolver = A2ACardResolver(http_client, base_url=f"http://127.0.0.1:{PORTA_SECRETARIA}")
    card_descoberto = await resolver.get_agent_card()

print_json(card_descoberto)

{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "Agente administrativo do curso: prazos, matrícula, certificado.",
  "name": "Secretaria",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Responde dúvidas administrativas do curso.",
      "id": "secretaria",
      "name": "Secretaria",
      "tags": [
        "curso",
        "administrativo"
      ]
    }
  ],
  "url": "http://127.0.0.1:8901/",
  "version": "0.1.0"
}


Repare que o card descoberto por HTTP é idêntico, campo a campo, ao `secretaria_card` que criamos na seção 3.2 — só que o cliente não precisou importar nenhum código do servidor para obtê-lo, apenas conhecer a URL.

Com o card em mãos, montamos o cliente e mandamos uma mensagem com `message/send` (síncrono: a chamada só retorna quando a task termina).

> `A2AClient` está marcado como *deprecated* nesta versão do SDK em favor de um `ClientFactory` mais genérico (que também fala gRPC, e escolhe transporte por preferência do card). Usamos `A2AClient` aqui porque é a forma mais direta de mostrar o protocolo, e ainda é o que a maioria dos tutoriais publicados usa — mas fica registrado para quando você for além deste notebook.

In [12]:
async with httpx.AsyncClient(timeout=30) as http_client:
    cliente_secretaria = A2AClient(httpx_client=http_client, agent_card=card_descoberto)

    mensagem = create_text_message_object(role=Role.user, content="Qual o prazo de matrícula?")
    requisicao = SendMessageRequest(id=str(uuid.uuid4()), params=MessageSendParams(message=mensagem))

    resposta = await cliente_secretaria.send_message(requisicao)
    task = resposta.root.result

print("state:", task.status.state)
print("resposta:", texto_da_resposta(task))

state: TaskState.completed
resposta: A data limite para matrícula no curso é **20 de agosto de 2026**. É importante garantir que a matrícula seja concluída até essa data para evitar restrições no acesso às disciplinas.


---

# 5. Streaming: observando o ciclo de vida em tempo real

`send_message` (seção 4) só devolve controle quando a task termina — para um agente que demora, o cliente fica sem visibilidade nenhuma até o fim. `send_message_streaming` troca isso por um fluxo de eventos: o cliente recebe cada `TaskStatusUpdateEvent` conforme o servidor os publica (via Server-Sent Events, por baixo do JSON-RPC), na ordem em que o `TaskUpdater` do executor os gerou.

É o mesmo problema que o **logbook** do Notebook 13 resolvia dentro de um processo — só que aqui a visibilidade atravessa a fronteira entre processos, porque faz parte do próprio protocolo.

In [13]:
async with httpx.AsyncClient(timeout=30) as http_client:
    cliente_secretaria = A2AClient(httpx_client=http_client, agent_card=card_descoberto)

    mensagem = create_text_message_object(role=Role.user, content="Qual o prazo de trancamento?")
    requisicao_streaming = SendStreamingMessageRequest(id=str(uuid.uuid4()), params=MessageSendParams(message=mensagem))

    print("Eventos recebidos, em ordem:")
    async for evento in cliente_secretaria.send_message_streaming(requisicao_streaming):
        resultado = evento.root.result
        if type(resultado).__name__ == "Task":
            print(f"  [Task criada]     state={resultado.status.state}")
        else:  # TaskStatusUpdateEvent
            texto_extra = ""
            if resultado.status.message is not None:
                texto_extra = " -- " + texto_da_resposta(Task(id="", context_id="", status=resultado.status))
            print(f"  [StatusUpdate]    state={resultado.status.state}  final={resultado.final}{texto_extra}")

Eventos recebidos, em ordem:
  [StatusUpdate]    state=TaskState.submitted  final=False
  [StatusUpdate]    state=TaskState.working  final=False


  [StatusUpdate]    state=TaskState.completed  final=True -- O prazo para trancamento do curso é até **30 de setembro de 2026**. 

Se você precisar de mais informações ou ajuda com outros prazos, estou à disposição! 😊


## Discussão

Repare na sequência: `submitted` → `working` → `completed`, exatamente os três eventos que `AgenteLangChainExecutor.execute` publicou via `updater.submit()` / `updater.start_work()` / `updater.complete()` (seção 3.1) — o cliente está literalmente vendo, em tempo real, cada chamada ao `TaskUpdater` do lado do servidor. `final=True` no último evento é o que diz ao cliente "pode parar de escutar".

---

# 6. Cancelando uma task em andamento

Tasks A2A são pensadas para trabalho que pode demorar — então o protocolo inclui `tasks/cancel` como cidadão de primeira classe, ao lado de `message/send`. Para observar isso com um resultado previsível (sem depender de quanto tempo o LLM local demora), criamos um segundo executor deliberadamente lento — não é um dos agentes do curso, só um dublê para este experimento.

In [14]:
class ExecutorLento(AgentExecutor):
    """Agente sintético que demora de propósito, só para demonstrar o cancelamento de uma task."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.submit()
        await updater.start_work()
        for _ in range(20):
            await asyncio.sleep(0.5)  # 10s de "trabalho" -- tempo de sobra para cancelarmos no meio
        await updater.complete(message=new_agent_text_message("terminei (não deveria chegar aqui)"))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        await updater.update_status(TaskState.canceled, final=True)


PORTA_LENTO = 8902

card_lento = AgentCard(
    name="AgenteLento",
    description="Agente sintético de teste, usado só para demonstrar cancelamento de task.",
    url=f"http://127.0.0.1:{PORTA_LENTO}/",
    version="0.1.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[AgentSkill(id="lento", name="lento", description="Demora de propósito.", tags=["teste"])],
)

handler_lento = DefaultRequestHandler(agent_executor=ExecutorLento(), task_store=InMemoryTaskStore())
app_lento = A2AStarletteApplication(agent_card=card_lento, http_handler=handler_lento).build()

subir_servidor_a2a(app_lento, PORTA_LENTO)
print(f"Servidor lento no ar em http://127.0.0.1:{PORTA_LENTO}")

Servidor lento no ar em http://127.0.0.1:8902


Mandamos a mensagem em modo **não bloqueante** (`blocking=False`) — assim `send_message` devolve a task assim que ela é criada (`state=submitted`), sem esperar `completed`. Isso nos dá um `task.id` para cancelar enquanto ela ainda está rodando.

In [15]:
async with httpx.AsyncClient(timeout=30) as http_client:
    resolver_lento = A2ACardResolver(http_client, base_url=f"http://127.0.0.1:{PORTA_LENTO}")
    card_lento_descoberto = await resolver_lento.get_agent_card()
    cliente_lento = A2AClient(httpx_client=http_client, agent_card=card_lento_descoberto)

    mensagem = create_text_message_object(role=Role.user, content="tarefa longa")
    requisicao = SendMessageRequest(
        id=str(uuid.uuid4()),
        params=MessageSendParams(message=mensagem, configuration={"blocking": False}),
    )
    resposta = await cliente_lento.send_message(requisicao)
    task_lenta = resposta.root.result
    print("Task criada:", task_lenta.id, "| state:", task_lenta.status.state)

    await asyncio.sleep(1)  # deixa a task começar a "trabalhar" antes de cancelar

    resposta_cancelamento = await cliente_lento.cancel_task(
        CancelTaskRequest(id=str(uuid.uuid4()), params=TaskIdParams(id=task_lenta.id))
    )
    print("Depois de cancel_task -> state:", resposta_cancelamento.root.result.status.state)

    confirmacao = await cliente_lento.get_task(
        GetTaskRequest(id=str(uuid.uuid4()), params=TaskQueryParams(id=task_lenta.id))
    )
    print("get_task confirma       -> state:", confirmacao.root.result.status.state)

Task criada: c35a6d21-96f9-4eac-8908-af8824b88ba1 | state: TaskState.submitted


Depois de cancel_task -> state: TaskState.canceled
get_task confirma       -> state: TaskState.canceled


## Discussão

`get_task` no final não é redundante: ele consulta o `TaskStore` do servidor de novo, de forma independente da resposta do `cancel_task`, então confirma que o cancelamento realmente persistiu do lado do servidor — e não só que o servidor *disse* que ia cancelar. É o mesmo cuidado do Padrão A (estado final) do Notebook 15: confiar no efeito registrado, não só na mensagem de quem executou.

---

# 7. Compondo agentes via A2A — o padrão *report*, agora entre processos

No Notebook 13, o padrão *report* tinha um supervisor que escolhia, a cada passo, qual especialista consultar — todos vivendo no mesmo `StateGraph`, no mesmo processo. Aqui reproduzimos a mesma ideia, com uma diferença estrutural: o "supervisor" é só um cliente A2A, e cada especialista é um **servidor A2A independente**. O supervisor não importa `agente_tutor` nem `agente_secretaria` — ele só conhece duas URLs.

Primeiro, subimos o `agente_tutor` como um segundo serviço A2A (a secretaria já está no ar desde a seção 3.3).

In [16]:
PORTA_TUTOR = 8903

tutor_card = AgentCard(
    name="Tutor",
    description="Agente técnico do curso: conteúdo e carga horária dos módulos.",
    url=f"http://127.0.0.1:{PORTA_TUTOR}/",
    version="0.1.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[
        AgentSkill(
            id="tutor",
            name="Tutor",
            description="Responde dúvidas sobre conteúdo e carga horária dos módulos.",
            tags=["curso", "conteúdo"],
        )
    ],
)

handler_tutor = DefaultRequestHandler(
    agent_executor=AgenteLangChainExecutor(agente_tutor, "Tutor"),
    task_store=InMemoryTaskStore(),
)
app_tutor = A2AStarletteApplication(agent_card=tutor_card, http_handler=handler_tutor).build()

subir_servidor_a2a(app_tutor, PORTA_TUTOR)
print(f"Servidor do Tutor no ar em http://127.0.0.1:{PORTA_TUTOR}")

Servidor do Tutor no ar em http://127.0.0.1:8903


Agora o "supervisor": descobre os dois AgentCards, decide qual dos dois agentes consultar (via saída estruturada do LLM — a mesma ideia do `DecisaoSupervisor` do Notebook 13, seção 4.1) e delega a pergunta pelo protocolo A2A.

In [17]:
class DecisaoSupervisorA2A(BaseModel):
    agente: Literal["Tutor", "Secretaria"] = Field(
        description="Qual agente remoto deve responder: Tutor (conteúdo/módulos) ou Secretaria (prazos/matrícula)."
    )


async def descobrir_agentes(http_client: httpx.AsyncClient, urls_base: dict[str, str]) -> dict[str, AgentCard]:
    cards = {}
    for nome, url_base in urls_base.items():
        resolver = A2ACardResolver(http_client, base_url=url_base)
        cards[nome] = await resolver.get_agent_card()
    return cards


async def perguntar_ao_supervisor_a2a(pergunta: str) -> None:
    urls_base = {"Tutor": f"http://127.0.0.1:{PORTA_TUTOR}", "Secretaria": f"http://127.0.0.1:{PORTA_SECRETARIA}"}

    async with httpx.AsyncClient(timeout=30) as http_client:
        cards = await descobrir_agentes(http_client, urls_base)

        roteador = llm.with_structured_output(DecisaoSupervisorA2A)
        decisao = roteador.invoke(
            "Você roteia perguntas de alunos para um de dois agentes remotos.\n"
            "- Tutor: conteúdo técnico e carga horária/duração de módulos específicos "
            "(ex.: 'o que é o módulo A7', 'quantas horas tem o A8').\n"
            "- Secretaria: prazos administrativos, matrícula, trancamento, certificado "
            "(não fala de conteúdo de módulo).\n\n"
            f"Pergunta do usuário: {pergunta}\n\nEscolha o agente."
        )

        card_escolhido = cards[decisao.agente]
        cliente = A2AClient(httpx_client=http_client, agent_card=card_escolhido)

        mensagem = create_text_message_object(role=Role.user, content=pergunta)
        requisicao = SendMessageRequest(id=str(uuid.uuid4()), params=MessageSendParams(message=mensagem))
        resposta = await cliente.send_message(requisicao)
        task = resposta.root.result

        print(f"Pergunta: {pergunta}")
        print(f"Supervisor delegou para: {decisao.agente}  (AgentCard: {card_escolhido.url})")
        print(f"Resposta ({task.status.state}): {texto_da_resposta(task)}")
        print()


await perguntar_ao_supervisor_a2a("Quanto tempo dura o módulo A7?")
await perguntar_ao_supervisor_a2a("Até quando posso emitir o certificado?")

Pergunta: Quanto tempo dura o módulo A7?
Supervisor delegou para: Tutor  (AgentCard: http://127.0.0.1:8903/)
Resposta (TaskState.completed): O módulo A7, cujo tema é **Arquiteturas Multiagente**, tem uma duração de **6 horas**.

Pergunta: Até quando posso emitir o certificado?
Supervisor delegou para: Secretaria  (AgentCard: http://127.0.0.1:8901/)
Resposta (TaskState.completed): O certificado pode ser emitido até 10 dias úteis após a conclusão do último módulo do curso. 

Precisa de ajuda para verificar se já está dentro do prazo?



## Discussão (comparando com o Notebook 13)

| | Padrão *report* do N13 | Este notebook (A2A) |
|---|---|---|
| Quem decide o próximo agente | LLM com saída estruturada, dentro do supervisor | idêntico — `DecisaoSupervisorA2A` cumpre o mesmo papel de `DecisaoSupervisor` |
| Onde os agentes vivem | nós do mesmo `StateGraph`, mesmo processo Python | processos HTTP independentes, cada um com seu próprio `AgentCard` |
| Como o supervisor conhece os agentes | referência direta ao objeto Python (`agente_tutor`) | descoberta em tempo de execução via `A2ACardResolver` |
| O que aconteceria se o Tutor fosse reescrito em outra linguagem | quebraria — o supervisor chama Python direto | nada muda para o supervisor — só o `AgentExecutor` do outro lado |
| Custo de implementação | baixo — uma função e um `Command` | maior — servidor HTTP, AgentCard, ciclo de vida de task por agente |

O ganho do A2A aqui não é fazer algo que o *report* não fazia — é o mesmo roteamento, com o mesmo tipo de decisão. O ganho é o que fica **desacoplado**: o supervisor não precisa mais compartilhar processo, linguagem ou deploy com quem responde.

---

# 8. A2A vs. MCP vs. padrões intra-processo — visão geral

Fechando o fio condutor da seção 1, agora com os três em mãos:

| | MCP (N5) | Padrões do N13 (memory/relay/report/debate) | A2A (este notebook) |
|---|---|---|---|
| Papel | agente consome ferramentas/dados | agentes coordenam dentro de um grafo | agentes conversam entre sistemas |
| Fronteira que cruza | processo → serviço de dados/ferramentas | nó → nó, mesmo processo | processo → processo, possivelmente organização → organização |
| Descoberta | `tools/list`, `resources/list` no servidor | nenhuma (o grafo já sabe) | `AgentCard`, em `/.well-known/agent-card.json` |
| Unidade de trabalho | chamada de tool (síncrona, curta) | passagem de nó no grafo | **task**, com estado, podendo ser assíncrona, cancelável, com streaming |
| Transporte típico | stdio (local) ou HTTP | chamada de função Python | HTTP + JSON-RPC (streaming via SSE) |
| Quando usar | seu agente precisa ler/escrever em um sistema externo | vários agentes, um dono, um processo, coordenação apertada | vários agentes, donos/times/frameworks diferentes, contrato público |

Os três **não são concorrentes** — um sistema real combina os três: um `StateGraph` (N13) cujos nós usam ferramentas via MCP (N5) e, quando a tarefa exige, delega para um agente externo via A2A (aqui). O `AgentExecutor` da seção 3.1, por exemplo, poderia perfeitamente chamar ferramentas MCP por dentro, antes de responder — A2A descreve a fronteira externa do agente, não o que ele faz por dentro dela.

---

# 9. Exercícios de fixação

## Exercício 1 — Expondo a Pesquisa via A2A

Suba `agente_pesquisa` (seção 0.4) como um terceiro servidor A2A, numa nova porta, seguindo o mesmo padrão da seção 3 (`AgenteLangChainExecutor` + `AgentCard` + `subir_servidor_a2a`). Confirme via `A2ACardResolver` que o AgentCard descoberto bate com o que você definiu.

## Exercício 2 — Supervisor com três agentes

Estenda `DecisaoSupervisorA2A` e `perguntar_ao_supervisor_a2a` (seção 7) para escolher entre três agentes (`Tutor`, `Secretaria`, `Pesquisa`, usando o servidor do Exercício 1), e teste com uma pergunta de conceito geral (ex.: "o que é um AgentCard?").

## Exercício 3 — `input-required`: quando o agente precisa perguntar de volta

Este notebook não usou o estado `input-required` (seção 2.3). Escreva um `AgentExecutor` cujo `execute` verifica se o pedido do usuário tem informação suficiente (ex.: um `emitir_declaracao` sem o nome do aluno) e, se não tiver, chama `updater.requires_input(...)` em vez de completar a task. Teste enviando uma mensagem incompleta e confira o `TaskState` resultante.

## Exercício 4 — Ciclo de vida sob falha

Modifique `AgenteLangChainExecutor.execute` para propagar um erro (ex.: force uma exceção) e capture-o para chamar `updater.failed(...)` em vez de deixar a exceção subir crua. Confirme, do lado do cliente, que `task.status.state == TaskState.failed` e que isso não derruba o servidor para as próximas requisições.

## Exercício 5 — A2A na versão mais nova

Rode `pip show a2a-sdk` depois de instalar sem fixar versão (`pip install -U a2a-sdk`, em um ambiente separado) e confirme que caiu na família `1.x`. Tente localizar, nessa versão, o caminho equivalente a `A2AStarletteApplication` (dica: `a2a.server.routes` tem funções de baixo nível para anexar rotas a um app FastAPI/Starlette existente, em vez de uma classe pronta). Anote a diferença — é o mesmo exercício de "ler o código-fonte instalado" que fizemos com a Eagle no Notebook 13.

## Exercício 6 — Qual camada usar?

Para cada situação, diga se você usaria MCP, um padrão intra-processo do Notebook 13, ou A2A — e por quê:

1. Seu agente precisa ler arquivos de um repositório Git local.
2. Três agentes seus, no mesmo time, revisam o mesmo texto em paralelo (gramática, clareza, tom).
3. Seu agente de atendimento precisa delegar a verificação de crédito a um agente mantido por outro departamento, com seu próprio ciclo de deploy.
4. Sua empresa quer permitir que agentes de parceiros externos descubram e usem um agente seu, sem acesso ao seu código.

---

# 10. Resumo da aula

Neste notebook, exploramos o **Agent2Agent (A2A)**, o protocolo para comunicação entre agentes que vivem em processos, frameworks ou organizações diferentes:

- **AgentCard**: o documento público de descoberta — quem o agente é, o que sabe fazer, onde falar com ele;
- **Message** e **Task**: a unidade de conteúdo e a unidade de trabalho, respectivamente — uma task tem um ciclo de vida (`TaskState`) pensado para trabalho assíncrono, cancelável e com necessidade de mais input;
- **`AgentExecutor`** + **`TaskUpdater`**: a interface que você implementa do lado do servidor, publicando eventos de status conforme o agente avança;
- **`A2ACardResolver`** + **`A2AClient`**: descoberta e conversa do lado do cliente, em modo síncrono (`send_message`) e streaming (`send_message_streaming`);
- **cancelamento** (`tasks/cancel`) como cidadão de primeira classe do protocolo, não um adendo;
- **composição**: o mesmo padrão *report* do Notebook 13, agora entre processos independentes, onde o supervisor só precisa de URLs, não de código compartilhado.

O ponto metodológico da aula: A2A não compete com MCP nem com os padrões intra-processo do Notebook 13 — cada um resolve uma fronteira diferente (agente↔ferramenta, agente↔agente-mesmo-processo, agente↔agente-processos-diferentes), e sistemas reais combinam os três.

## 10.1 Checklist de compreensão

Antes de seguir, você deve conseguir responder:

1. Por que uma task A2A tem estado (`TaskState`), e uma chamada de tool MCP não precisa ter?
2. O que o `AgentCard` resolve que uma simples URL fixa no código não resolveria?
3. Qual a diferença entre `send_message` e `send_message_streaming` — e o que você perde ao usar o primeiro num agente de longa duração?
4. Por que o supervisor da seção 7 não precisa importar `agente_tutor` nem `agente_secretaria`?
5. Em que ponto A2A e MCP poderiam ser usados **juntos**, dentro do mesmo agente?

## 10.2 Próximos passos

A partir daqui, os próximos passos naturais são:

- trocar `InMemoryTaskStore` por um `DatabaseTaskStore`, para tasks sobreviverem a um reinício do servidor;
- explorar `push_notification` — o servidor A2A avisando o cliente por webhook quando uma task muda de estado, em vez do cliente precisar ficar com uma conexão de streaming aberta;
- migrar o cliente para o `ClientFactory` (mencionado na seção 4), que negocia transporte a partir das preferências do `AgentCard` em vez de assumir JSON-RPC;
- combinar A2A com o padrão *debate* do Notebook 13: dois agentes A2A de organizações diferentes debatendo, com um moderador local julgando as respostas.

---

# 11. Referências

- Agent2Agent (A2A) — especificação e documentação: https://a2a-protocol.org/latest/specification/
- A2A — repositório do Python SDK: https://github.com/a2aproject/a2a-python
- A2A — pacote no PyPI: https://pypi.org/project/a2a-sdk/
- Model Context Protocol — para contraste (Notebook 5): https://modelcontextprotocol.io/
- Padrões de comunicação intra-processo — Notebook 13 deste curso.